# Calendar Table — dbr_caresync.silver.calendar

Generates a standard calendar/date dimension table spanning **1940-01-01 to 2030-12-31** with all common date attributes including keys, names, flags, and period boundaries.

In [0]:
from pyspark.sql import functions as F

# ── 1. Generate date spine ───────────────────────────────────────────────────
df = spark.sql("""
    SELECT explode(
        sequence(DATE '1940-01-01', DATE '2030-12-31', INTERVAL 1 DAY)
    ) AS date
""")

# ── 2. Derive all calendar columns ──────────────────────────────────────────
calendar_df = df.select(

    # --- Primary key & date ---
    F.date_format("date", "yyyyMMdd").cast("int").alias("date_key"),
    F.col("date"),

    # --- Year-level ---
    F.year("date").alias("year"),
    F.trunc("date", "year").alias("first_day_of_year"),
    F.make_date(F.year("date"), F.lit(12), F.lit(31)).alias("last_day_of_year"),
    F.when(
        ((F.year("date") % 4 == 0) & (F.year("date") % 100 != 0)) |
        (F.year("date") % 400 == 0), True
    ).otherwise(False).alias("is_leap_year"),

    # --- Quarter-level ---
    F.quarter("date").alias("quarter"),
    F.concat(F.lit("Q"), F.quarter("date").cast("string")).alias("quarter_name"),
    F.concat(F.year("date").cast("string"), F.lit("-Q"), F.quarter("date").cast("string")).alias("year_quarter"),
    F.trunc("date", "quarter").alias("first_day_of_quarter"),
    F.date_add(F.add_months(F.trunc("date", "quarter"), 3), -1).alias("last_day_of_quarter"),

    # --- Month-level ---
    F.month("date").alias("month"),
    F.date_format("date", "MM").alias("month_padded"),
    F.date_format("date", "MMMM").alias("month_name"),
    F.date_format("date", "MMM").alias("month_name_short"),
    F.date_format("date", "yyyy-MM").alias("year_month"),
    F.trunc("date", "month").alias("first_day_of_month"),
    F.last_day("date").alias("last_day_of_month"),
    F.datediff(F.last_day("date"), F.trunc("date", "month")).alias("days_in_month"),

    # --- Week-level ---
    F.weekofyear("date").alias("week_of_year"),
    # ISO week: Monday=1 ... Sunday=7  (Spark dayofweek: Sun=1 ... Sat=7)
    ((F.dayofweek("date") + 5) % 7 + 1).alias("day_of_week_iso"),
    F.date_trunc("week", F.col("date")).cast("date").alias("week_start_date"),           # Monday
    F.date_add(F.date_trunc("week", F.col("date")).cast("date"), 6).alias("week_end_date"),  # Sunday

    # --- Day-level ---
    F.dayofmonth("date").alias("day_of_month"),
    F.date_format("date", "dd").alias("day_of_month_padded"),
    F.dayofyear("date").alias("day_of_year"),
    F.dayofweek("date").alias("day_of_week"),           # 1=Sunday ... 7=Saturday (Spark default)
    F.date_format("date", "EEEE").alias("day_name"),
    F.date_format("date", "EEE").alias("day_name_short"),
    F.when(F.dayofweek("date").isin(1, 7), True).otherwise(False).alias("is_weekend"),
    F.when(F.dayofweek("date").isin(1, 7), False).otherwise(True).alias("is_weekday"),
)

# ── 3. Write to Unity Catalog ────────────────────────────────────────────────
(
    calendar_df
    .write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("dbr_caresync.silver.calendar")
)

print("✅ Calendar table created: dbr_caresync.silver.calendar")
print(f"   Rows: {spark.table('dbr_caresync.silver.calendar').count():,}")
print(f"   Columns: {len(calendar_df.columns)}")
display(spark.table("dbr_caresync.silver.calendar").limit(5))